**Objective:** model and interpret the drivers of country-level mobile money adoption rates (`mobileaccount_t_d`).

In [5]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling & stats
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Diagnostics
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Quick plotting style
sns.set_theme(style="whitegrid")


In [7]:
# Load data
df = pd.read_csv('../data/processed/global_findex_cleaned.csv')
df.head()

,countrynewwb,codewb,year,pop_adult,regionwb24_hi,incomegroupwb24,group,group2,account_t_d,fiaccount_t_d,...,con18,con19,internet,con26d,con27,con30a,con30b,con30c,con30d,con30e
0,Afghanistan,AFG,2011,14575546.0,South Asia (excluding high income),Low income,all,all,0.090050,0.090050,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Albania,ALB,2011,2281010.0,Europe & Central Asia (excluding high income),Upper middle income,all,all,0.282681,0.282681,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Algeria,DZA,2011,26251587.0,Middle East & North Africa (excluding high inc...,Lower middle income,all,all,0.332861,0.332861,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Angola,AGO,2011,12779501.0,Sub-Saharan Africa (excluding high income),Lower middle income,all,all,0.392035,0.392035,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Argentina,ARG,2011,30685516.0,Latin America & Caribbean (excluding high income),Upper middle income,all,all,0.331302,0.331302,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Quick check (shape / columns / dtypes / missing)
print("shape:", df.shape)
print("\nColumns (sample):", df.columns.tolist()[:40])
print("\nMissing per columns (top 20):")
print(df.isnull().sum().sort_values(ascending=False).head(20))
df.describe(include='all').T

shape: (7880, 126)

Columns (sample): ['countrynewwb', 'codewb', 'year', 'pop_adult', 'regionwb24_hi', 'incomegroupwb24', 'group', 'group2', 'account_t_d', 'fiaccount_t_d', 'mobileaccount_t_d', 'borrow_any_t_d', 'fin4_d', 'dig_acc', 'fin26a', 'fin26b', 'fin27a', 'fin17a_17a1_d', 'fin17a', 'fin17b', 'fin17c', 'fin22d', 'fin22e', 'fin22a_22a1_22g_d', 'fin22a', 'fin22b', 'fin22c', 'fin24sav', 'fin24fam', 'fin24work', 'fin24bor', 'fin24sell', 'fin24aVD', 'fin24aSD', 'fin24aND', 'fin24aSD_ND', 'fin24aP', 'fin24aN', 'fin24sav_SD_ND', 'fin24fam_SD_ND']

Missing per columns (top 20):
fin24bor    7078
fh2a        7073
fin6m       7049
fin45e      7029
fin9b       7013
con26d      7004
con30e      6992
fin5m       6989
fin22h      6983
fin9a       6977
fin22c      6976
fin17b      6976
con30d      6956
con30a      6956
con30b      6956
fin8        6953
con27       6932
fin34b      6932
con30c      6920
con11       6908
dtype: int64


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
countrynewwb,7880,162,Malawi,57,NaN,NaN,NaN,NaN,NaN,NaN,NaN
codewb,7880,162,MWI,57,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,7880.0,NaN,NaN,NaN,2017.762563,4.700454,2011.0,2014.0,2017.0,2022.0,2024.0
pop_adult,7880.0,NaN,NaN,NaN,38015962.817893,126950906.086927,228016.0,3739546.0,8517166.0,25903954.0,1176678702.0
regionwb24_hi,7880,7,High income,2630,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
con30a,924.0,NaN,NaN,NaN,0.478057,0.208324,0.043145,0.311948,0.476158,0.634895,0.976289
con30b,924.0,NaN,NaN,NaN,0.455369,0.196877,0.05776,0.297876,0.449223,0.589511,0.966377
con30c,960.0,NaN,NaN,NaN,0.518547,0.230421,0.047502,0.330125,0.548915,0.705397,0.984643
con30d,924.0,NaN,NaN,NaN,0.44119,0.205357,0.042529,0.270446,0.438448,0.598149,0.942952


# Choose target & candidate predictors.
**Target (y):** `mobileaccount_t_d` (proportion 0-1)
**Candidate predictors (x):**
- `account_t_d`
- `year`
- `regionwb24_hi` (region)
- `incomegroupwb24` (income group)
- `pop_adult`


In [9]:
# Select relevant columns
cols = ['year', 'regionwb24_hi', 'incomegroupwb24', 'pop_adult',
        'account_t_d', 'mobileaccount_t_d']
df_model = df[cols].copy()
print("shape:", df_model.shape)
df_model.head()

shape: (7880, 6)


,year,regionwb24_hi,incomegroupwb24,pop_adult,account_t_d,mobileaccount_t_d
0,2011,South Asia (excluding high income),Low income,14575546.0,0.090050,NaN
1,2011,Europe & Central Asia (excluding high income),Upper middle income,2281010.0,0.282681,NaN
2,2011,Middle East & North Africa (excluding high inc...,Lower middle income,26251587.0,0.332861,NaN
3,2011,Sub-Saharan Africa (excluding high income),Lower middle income,12779501.0,0.392035,NaN
4,2011,Latin America & Caribbean (excluding high income),Upper middle income,30685516.0,0.331302,NaN


In [10]:
# Handle missing data

# Show how many rows have missing target
print("Rows with missing target:", df_model['mobileaccount_t_d'].isnull().sum())

# Inspect missing pattern for predictors
print("\nMissing per columns:")
print(df_model.isnull().sum().sort_values(ascending=False))

Rows with missing target: 5654

Missing per columns:
mobileaccount_t_d    5654
account_t_d            90
year                    0
regionwb24_hi           0
incomegroupwb24         0
pop_adult               0
dtype: int64


In [11]:
# Impute missing values in the target variable
df_model['mobileaccount_t_d'].fillna(method='bfill', axis=0).fillna(0)

C:\Users\shari\AppData\Local\Temp\ipykernel_19388\2692350365.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_model['mobileaccount_t_d'].fillna(method='bfill', axis=0).fillna(0)


0       0.003044
1       0.003044
2       0.003044
3       0.003044
4       0.003044
          ...   
7875    0.823123
7876    0.823123
7877    0.572759
7878    0.522832
7879    0.290422
Name: mobileaccount_t_d, Length: 7880, dtype: float64

: 